# Day 3 — Train the SATB Harmony Model (Colab)

Two-phase training on Google Colab T4.

- **Phase A** — pretrain hybrid model on JSB chorales (~2 h)
- **Phase B** — fine-tune on jaCappella from Phase A checkpoint (~1 h)

Both phases are resume-safe: if Colab disconnects, re-run every cell from the top. `last.pt` in the Drive checkpoint dir wins over `--init-from`, so Phase B won't accidentally restart from scratch.

Operator runbook lives at `docs/specs/training_runs.md`.

## 1. Clone the repo + install dependencies

Idempotent — re-running is a no-op if the repo is already cloned.

In [ ]:
%cd /content
![ -d acapella-arranger ] || git clone https://github.com/jessyy2006/acapella-arranger.git
%cd acapella-arranger
!git fetch --all --quiet && git checkout main && git pull --ff-only
!pip install -q -r requirements.txt

## 2. Authenticate with HuggingFace

jaCappella is a gated dataset. Paste a read token from https://huggingface.co/settings/tokens and make sure you've accepted the license at https://huggingface.co/datasets/jaCappella/jaCappella first.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Download raw corpora + build train/val/test splits

`prepare_data.py` writes 9 files to `data/processed/` — combined, JSB-only, and jaCappella-only splits. Training configs point at the phase-specific files.

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py

## 4. Mount Google Drive for checkpoint persistence

Colab's free tier disconnects after ~90 min idle or ~12 h continuous. Checkpoints on the mounted Drive survive; checkpoints on the ephemeral Colab VM don't.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aca-adapt/checkpoints
!mkdir -p /content/drive/MyDrive/aca-adapt/reports
# Symlink Drive dirs into the repo so the trainer's relative paths land on Drive.
!rm -rf checkpoints reports
!ln -s /content/drive/MyDrive/aca-adapt/checkpoints checkpoints
!ln -s /content/drive/MyDrive/aca-adapt/reports reports
!ls -la checkpoints reports

## 5. Phase A — JSB pretrain (~2 h on T4)

Resumes automatically from `checkpoints/phase_a/last.pt` if Colab disconnects mid-run.

In [ ]:
!python -m src.training.train --config configs/train.yaml --phase pretrain

## 6. Phase B — jaCappella fine-tune (~1 h on T4)

Picks up from Phase A's `best.pt`. If interrupted, `checkpoints/phase_b/last.pt` wins over `--init-from` so you don't restart from Phase A weights.

In [ ]:
!python -m src.training.train \
  --config configs/train.yaml \
  --phase finetune \
  --init-from checkpoints/phase_a/best.pt

## 7. Ablation runs (optional — run after both phases land)

Re-use the same CLI with different `--model-class` / `--phase` combos. Each run writes into its own `checkpoints/<run_name>/` subdirectory so they don't collide.

In [ ]:
# Baseline pretrain (ablation: LSTM vs Transformer decoder)
!python -m src.training.train --config configs/train.yaml --phase pretrain --model-class baseline

# Baseline fine-tune
!python -m src.training.train \
  --config configs/train.yaml \
  --phase finetune \
  --model-class baseline \
  --init-from checkpoints/phase_a/best.pt